# Music Annotation Project V8.2
## Development Whole-Page Experiment

**Runtime: GPU.** Select a T4 GPU, then Run all.

V8.2 is a fresh **development-only** experiment. It does not reconstruct the historical V8.0/V8.1 lock bundle. It retrains the unchanged 33x33 spatial observed-print CNN under source-group OOF, fits and persists fold-specific cross-fitted isotonic calibration, scores every eligible held-out pixel, builds complete corrected pages, and evaluates them against V7.8.

**Fixed contract**
- authoritative mask only;
- clean truth for labels/scoring only, never an inference channel;
- source-group OOF;
- 11 observable channels, 33x33 patches;
- PRINT 0.995, ANNOTATION 0.995, PRINT margin 0.990;
- singleton PRINT demoted to UNCERTAIN;
- V8.2 equals V7.8 everywhere except final PRINT coordinates, where the exact observed byte is copied.

This run deliberately does **not** claim independent validation or production promotion. It snapshots its own inputs and saves every learned model/calibrator so this run can be frozen later if the evidence warrants it.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy scikit-learn pyarrow torch torchvision matplotlib joblib


## 1. Configuration and run identity


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, sys, platform, gc, shutil, time
import cv2, torch, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.isotonic import IsotonicRegression
ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
MANIFEST_PATH=ROOT/'development_score_manifest.csv'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_2'
SEED=42; PATCH=33; HALF=16; DARK_THRESHOLD=200; MAX_PER_PAGE_CLASS=25000
BATCH=512; EPOCHS=12; LR=1e-3; WEIGHT_DECAY=1e-4
PRINT_THRESHOLD=.995; ANNOTATION_THRESHOLD=.995; PRINT_MARGIN=.990; BOOTSTRAP_RESAMPLES=10000
CHANNELS=['obs','norm','mask','depth','horizontal','vertical','compact','preserve','remove','ambiguous','edge']
PAGE_ALLOWLIST=None  # optional smoke test only
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE!='cuda': raise RuntimeError('V8.2 requires a GPU runtime. Select a T4 GPU and rerun.')
torch.use_deterministic_algorithms(True); torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_v8_2_development_whole_page'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; MODELS=OUT/'models'; CALIBRATORS=OUT/'calibrators'; WHOLE=OUT/'whole_pages'; AUDIT=OUT/'audit'; VISUAL=OUT/'visual_audit'
for p in [OUT,INPUTS,TABLES,MODELS,CALIBRATORS,WHOLE,AUDIT,VISUAL]: p.mkdir(parents=True,exist_ok=False)
print('GPU:',torch.cuda.get_device_name(0)); print('Output:',OUT)


## 2. Helpers, current manifest snapshot and V7.5 reference


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def latest(root,required):
 hits=[]
 if Path(root).exists():
  for p in Path(root).rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required): hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''): h.update(b)
 return h.hexdigest()
def object_hash(x): return hashlib.sha256(json.dumps(x,sort_keys=True,default=str,separators=(',',':')).encode()).hexdigest()
def array_hash(a):
 a=np.ascontiguousarray(a); h=hashlib.sha256(); h.update(str(a.dtype).encode()); h.update(str(a.shape).encode()); h.update(a.tobytes()); return h.hexdigest()
def reset_seed(seed=SEED): random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
def wilson_upper(errors,n,z=1.96):
 if n==0:return np.nan
 p=errors/n; d=1+z*z/n; c=p+z*z/(2*n); r=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n)); return (c+r)/d
if not MANIFEST_PATH.exists(): raise RuntimeError(f'Missing {MANIFEST_PATH}')
manifest=pd.read_csv(MANIFEST_PATH,dtype=str)
missing={'page','source_group'}-set(manifest.columns)
if missing: raise RuntimeError(f'Manifest missing required columns: {sorted(missing)}')
if manifest.page.duplicated().any(): raise RuntimeError('development_score_manifest.csv contains duplicate page values.')
if manifest.source_group.isna().any() or (manifest.source_group.astype(str).str.strip()=='').any(): raise RuntimeError('Manifest contains missing source_group values.')
shutil.copy2(MANIFEST_PATH,INPUTS/'development_score_manifest.csv')
V75=latest(V75_ROOT,['tables/cache_status.csv'])
if V75 is None: raise RuntimeError('Complete V7.5 development run not found.')
status=pd.read_csv(V75/'tables'/'cache_status.csv'); cache={str(r.page):Path(r.cache) for _,r in status.iterrows()}
clean_map=image_map(ROOT/'source'/'clean'); observed_map=image_map(ROOT/'aligned')
available=set(cache)&set(clean_map)&set(observed_map); manifest_pages=set(manifest.page.astype(str)); pages=sorted(manifest_pages&available)
if PAGE_ALLOWLIST is not None: pages=[p for p in pages if p in set(PAGE_ALLOWLIST)]
if not pages: raise RuntimeError('No development pages remain after manifest/V7.5/aligned intersection.')
if PAGE_ALLOWLIST is None and set(pages)!=manifest_pages:
 raise RuntimeError(f'Current manifest inputs unavailable: {sorted(manifest_pages-available)}')
manifest=manifest[manifest.page.astype(str).isin(pages)].copy(); manifest['page']=manifest.page.astype(str); manifest['source_group']=manifest.source_group.astype(str); meta=manifest.set_index('page')
print('V7.5:',V75); print('Pages:',len(pages),'Source groups:',manifest.source_group.nunique()); print('Manifest SHA:',sha256_file(INPUTS/'development_score_manifest.csv'))


## 3. Labels and observable channels


In [ ]:
def printed_ink(clean): return cv2.adaptiveThreshold(clean,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def truth_maps(clean):
 ink=printed_ink(clean); k=np.ones((3,3),np.uint8); near=cv2.dilate(ink.astype(np.uint8),k)>0; eroded=cv2.erode(ink.astype(np.uint8),k)>0; return ink,near,np.logical_xor(near,eroded)
def background_surface(obs,mask,sigma=45.0):
 vis=(~mask)&(obs>=200); num=cv2.GaussianBlur(obs.astype(np.float32)*vis,(0,0),sigma); den=cv2.GaussianBlur(vis.astype(np.float32),(0,0),sigma); fallback=cv2.GaussianBlur(obs,(0,0),sigma); return np.clip(np.where(den>.02,num/np.maximum(den,1e-4),fallback),0,255).astype(np.uint8)
runtime={}; input_audit=[]; req=['mask','geometry_t80','g80_preserve','g80_remove','g80_ambiguous','g80_horizontal','g80_vertical','g80_compact']
for page in pages:
 with np.load(cache[page]) as z:
  absent=[k for k in req if k not in z.files]
  if absent: raise RuntimeError(f'{page}: V7.5 cache missing {absent}')
  a={k:z[k] for k in req}
 clean=cv2.imread(str(clean_map[page]),0); obs=cv2.imread(str(observed_map[page]),0)
 if clean is None or obs is None or clean.shape!=obs.shape or any(v.shape!=obs.shape for v in a.values()): raise RuntimeError(f'{page}: unreadable input or shape mismatch')
 mask=a['mask']>0; baseline=a['geometry_t80']
 if not np.array_equal(baseline[~mask],obs[~mask]): raise RuntimeError(f'{page}: baseline outside-mask mismatch')
 ink,near,boundary=truth_maps(clean); eligible=mask&(obs<DARK_THRESHOLD)&~(a['g80_preserve']>0)
 label=np.full(obs.shape,-1,np.int8); label[eligible&near&~boundary]=1; label[eligible&~near&~boundary]=0; label[eligible&boundary]=2
 paper=background_surface(obs,mask); norm=cv2.divide(obs,np.maximum(paper,1),scale=255)
 runtime[page]={'obs':obs,'mask':mask,'baseline':baseline,'label':label,'norm':norm,'depth':cv2.distanceTransform(mask.astype(np.uint8),cv2.DIST_L2,5),'edge':cv2.Canny(obs,70,170)>0,'preserve':a['g80_preserve']>0,'remove':a['g80_remove']>0,'ambiguous':a['g80_ambiguous']>0,'horizontal':a['g80_horizontal']>0,'vertical':a['g80_vertical']>0,'compact':a['g80_compact']>0}
 input_audit.append({'page':page,'source_group':meta.loc[page,'source_group'],'eligible':int(eligible.sum()),'print':int((label==1).sum()),'annotation':int((label==0).sum()),'truth_uncertain':int((label==2).sum()),'cache_sha256':sha256_file(cache[page]),'observed_sha256':sha256_file(observed_map[page]),'clean_sha256':sha256_file(clean_map[page])})
pd.DataFrame(input_audit).to_csv(AUDIT/'input_and_label_audit.csv',index=False)
(INPUTS/'channel_schema.json').write_text(json.dumps({'patch':PATCH,'channels':CHANNELS,'dtype':'float32','normalisation':{'obs':'divide_255','norm':'divide_255','other':'native_float32'},'padding':'reflect'},indent=2))
print(pd.DataFrame(input_audit)[['eligible','print','annotation','truth_uncertain']].sum())


## 4. Stable roster, all eligible pixels and source-group folds


In [ ]:
rng=np.random.default_rng(SEED); stable_rows=[]; all_rows=[]
for page,z in runtime.items():
 group=str(meta.loc[page,'source_group'])
 for cls in [0,1,2]:
  ys,xs=np.where(z['label']==cls)
  all_rows.extend({'page':page,'source_group':group,'y':int(y),'x':int(x),'truth':int(cls)} for y,x in zip(ys,xs))
 for cls in [0,1]:
  ys,xs=np.where(z['label']==cls); idx=np.arange(len(ys)); take=idx if len(idx)<=MAX_PER_PAGE_CLASS else rng.choice(idx,MAX_PER_PAGE_CLASS,replace=False)
  stable_rows.extend({'page':page,'source_group':group,'y':int(ys[j]),'x':int(xs[j]),'truth':int(cls)} for j in take)
stable=pd.DataFrame(stable_rows).sort_values(['source_group','page','y','x']).reset_index(drop=True); all_candidates=pd.DataFrame(all_rows).sort_values(['source_group','page','y','x']).reset_index(drop=True)
if stable.empty or all_candidates.empty: raise RuntimeError('Stable or all-eligible roster is empty.')
stable['sample_id']=np.arange(len(stable)); all_candidates['candidate_id']=np.arange(len(all_candidates))
groups=stable.source_group; splitter=LeaveOneGroupOut() if groups.nunique()<=10 else GroupKFold(n_splits=min(10,groups.nunique())); fold=np.full(len(stable),-1,int); group_to_fold={}; split_rows=[]
for f,(tr,te) in enumerate(splitter.split(stable,stable.truth,groups),1):
 train_g=set(groups.iloc[tr]); test_g=set(groups.iloc[te]); overlap=train_g&test_g
 if overlap: raise RuntimeError(f'Fold {f}: source-group leakage {sorted(overlap)}')
 fold[te]=f
 for g in test_g:
  if g in group_to_fold: raise RuntimeError(f'Source group {g} assigned twice')
  group_to_fold[g]=f
 split_rows.append({'fold':f,'train_groups':len(train_g),'test_groups':len(test_g),'overlap_groups':0,'test_rows':len(te),'test_pages':stable.iloc[te].page.nunique(),'test_source_groups':'|'.join(sorted(test_g))})
stable['fold']=fold; all_candidates['fold']=all_candidates.source_group.map(group_to_fold)
if (stable.fold<1).any() or all_candidates.fold.isna().any(): raise RuntimeError('Incomplete fold assignment.')
all_candidates['fold']=all_candidates.fold.astype(int)
stable.to_csv(TABLES/'stable_sample_manifest.csv',index=False); all_candidates.to_parquet(TABLES/'eligible_pixel_manifest.parquet',index=False); pd.DataFrame(split_rows).to_csv(TABLES/'source_group_split_audit.csv',index=False); pd.DataFrame([{'source_group':g,'fold':f} for g,f in sorted(group_to_fold.items())]).to_csv(INPUTS/'fold_map.csv',index=False)
print('Stable:',len(stable),'Eligible:',len(all_candidates),'Folds:',sorted(stable.fold.unique()))


## 5. Tensor builder and compact CNN


In [ ]:
padded={}
for page,z in runtime.items():
 padded[page]={}
 for key in CHANNELS:
  a=z[key].astype(np.float32)
  if key in ['obs','norm']: a/=255.0
  padded[page][key]=np.pad(a,HALF,mode='reflect')
page_list=pages; page_index={p:i for i,p in enumerate(page_list)}
def encode(df): return df.page.map(page_index).to_numpy(np.int16),df.y.to_numpy(np.int32),df.x.to_numpy(np.int32),df.truth.to_numpy(np.int8)
stable_pid,stable_y,stable_x,stable_truth=encode(stable); all_pid,all_y,all_x,all_truth=encode(all_candidates)
def get_patch(pid,y,x):
 page=page_list[int(pid)]; return np.stack([padded[page][k][y:y+PATCH,x:x+PATCH] for k in CHANNELS]).astype(np.float32)
class ArrayPatchDataset(Dataset):
 def __init__(self,idx,pids,ys,xs,truth): self.idx=np.asarray(idx); self.pids=pids; self.ys=ys; self.xs=xs; self.truth=truth
 def __len__(self): return len(self.idx)
 def __getitem__(self,i):
  j=int(self.idx[i]); return torch.from_numpy(get_patch(self.pids[j],self.ys[j],self.xs[j])),torch.tensor(int(self.truth[j]),dtype=torch.long)
class TopologyCNN(nn.Module):
 def __init__(self,nc=len(CHANNELS)):
  super().__init__(); self.net=nn.Sequential(nn.Conv2d(nc,24,3,padding=1),nn.ReLU(),nn.BatchNorm2d(24),nn.MaxPool2d(2),nn.Conv2d(24,48,3,padding=1),nn.ReLU(),nn.BatchNorm2d(48),nn.MaxPool2d(2),nn.Conv2d(48,64,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1)); self.head=nn.Linear(64,2)
 def forward(self,x): return self.head(self.net(x).flatten(1))
def worker_init(w): random.seed(SEED+w); np.random.seed(SEED+w)
def loader(idx,pids,ys,xs,truth,shuffle,seed):
 g=torch.Generator(); g.manual_seed(seed); return DataLoader(ArrayPatchDataset(idx,pids,ys,xs,truth),batch_size=BATCH,shuffle=shuffle,num_workers=2,pin_memory=True,generator=g,worker_init_fn=worker_init,persistent_workers=False)
def fit_fold(train_idx):
 reset_seed(); model=TopologyCNN().to(DEVICE); opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WEIGHT_DECAY); counts=np.bincount(stable_truth[train_idx],minlength=2); weights=torch.tensor(counts.sum()/np.maximum(counts,1),dtype=torch.float32,device=DEVICE); loss_fn=nn.CrossEntropyLoss(weight=weights); hist=[]
 for ep in range(EPOCHS):
  total=0.; model.train()
  for xb,yb in loader(train_idx,stable_pid,stable_y,stable_x,stable_truth,True,SEED):
   xb=xb.to(DEVICE,non_blocking=True); yb=yb.to(DEVICE,non_blocking=True); opt.zero_grad(set_to_none=True); loss=loss_fn(model(xb),yb); loss.backward(); opt.step(); total+=loss.detach().item()*len(yb)
  hist.append(total/len(train_idx)); print(f'  epoch {ep+1:02d}/{EPOCHS}: loss={hist[-1]:.6f}')
 return model,hist
def predict(model,idx,pids,ys,xs,truth):
 out=[]; model.eval()
 with torch.no_grad():
  for xb,_ in loader(idx,pids,ys,xs,truth,False,SEED): out.extend(torch.softmax(model(xb.to(DEVICE,non_blocking=True)),1)[:,1].cpu().numpy())
 return np.asarray(out,dtype=np.float64)
print('Tensor/CNN ready.')


## 6. OOF training and scoring: models are persisted


In [ ]:
raw_stable=np.full(len(stable),np.nan); raw_all=np.full(len(all_candidates),np.nan); model_rows=[]; history_rows=[]
for f in sorted(stable.fold.unique()):
 print('\n'+'='*80+'\nFOLD',f); tr=np.where(stable.fold.to_numpy()!=f)[0]; te=np.where(stable.fold.to_numpy()==f)[0]; ae=np.where(all_candidates.fold.to_numpy()==f)[0]; t0=time.time(); model,hist=fit_fold(tr); raw_stable[te]=predict(model,te,stable_pid,stable_y,stable_x,stable_truth); raw_all[ae]=predict(model,ae,all_pid,all_y,all_x,all_truth); path=MODELS/f'fold{int(f)}_seed{SEED}.pt'; torch.save(model.state_dict(),path); model_rows.append({'fold':int(f),'checkpoint':str(path),'checkpoint_sha256':sha256_file(path),'train_rows':len(tr),'stable_test_rows':len(te),'eligible_test_rows':len(ae),'seconds':time.time()-t0}); history_rows.extend({'fold':int(f),'epoch':i+1,'mean_loss':v} for i,v in enumerate(hist)); del model; gc.collect(); torch.cuda.empty_cache()
if not np.isfinite(raw_stable).all() or not np.isfinite(raw_all).all(): raise RuntimeError('Incomplete raw OOF predictions.')
np.save(TABLES/'stable_raw_oof.npy',raw_stable); np.save(TABLES/'all_eligible_raw_oof.npy',raw_all); pd.DataFrame(model_rows).to_csv(AUDIT/'model_inventory.csv',index=False); pd.DataFrame(history_rows).to_csv(TABLES/'training_history.csv',index=False)


## 7. Cross-fitted isotonic calibration: calibrators are persisted


In [ ]:
p_stable=np.full(len(stable),np.nan); p_all=np.full(len(all_candidates),np.nan); calibrator_rows=[]
for f in sorted(stable.fold.unique()):
 tr=np.where(stable.fold.to_numpy()!=f)[0]; te=np.where(stable.fold.to_numpy()==f)[0]; ae=np.where(all_candidates.fold.to_numpy()==f)[0]; iso=IsotonicRegression(out_of_bounds='clip').fit(raw_stable[tr],stable_truth[tr]); p_stable[te]=iso.predict(raw_stable[te]); p_all[ae]=iso.predict(raw_all[ae]); path=CALIBRATORS/f'fold{int(f)}_seed{SEED}_isotonic.joblib'; joblib.dump(iso,path); payload={'x':iso.X_thresholds_.tolist(),'y':iso.y_thresholds_.tolist()}; side={'fold':int(f),'seed':SEED,'method':'IsotonicRegression','out_of_bounds':'clip','training_rule':'stable OOF rows outside held-out fold','semantic_sha256':object_hash(payload),'joblib_sha256':sha256_file(path),'x_thresholds':payload['x'],'y_thresholds':payload['y']}; (CALIBRATORS/f'fold{int(f)}_seed{SEED}_isotonic.json').write_text(json.dumps(side,indent=2)); replay=joblib.load(path).predict(raw_all[ae]).astype(np.float64)
 if not np.array_equal(replay,p_all[ae]): raise RuntimeError(f'Fold {f}: saved calibrator reload mismatch')
 calibrator_rows.append({'fold':int(f),'joblib':str(path),'joblib_sha256':sha256_file(path),'semantic_sha256':object_hash(payload),'exact_reload_replay':True})
if not np.isfinite(p_stable).all() or not np.isfinite(p_all).all(): raise RuntimeError('Incomplete calibrated OOF probabilities.')
np.save(TABLES/'stable_calibrated_oof.npy',p_stable); np.save(TABLES/'all_eligible_calibrated_oof.npy',p_all); pd.DataFrame(calibrator_rows).to_csv(AUDIT/'calibrator_inventory.csv',index=False); print('Calibrators persisted:',len(calibrator_rows))


## 8. Decision rule and page-global no-singleton rule


In [ ]:
def decide(prob):
 p=np.asarray(prob); q=1-p; out=np.full(len(p),'UNCERTAIN',object); out[q>=ANNOTATION_THRESHOLD]='ANNOTATION'; out[(p>=PRINT_THRESHOLD)&((p-q)>=PRINT_MARGIN)]='PRINT'; return out
full=all_candidates.copy(); full['raw_print']=raw_all; full['p_print']=p_all; full['decision_pre_run_rule']=decide(p_all); full['decision']=full.decision_pre_run_rule.copy(); singleton_demotions=0
for page,g in full.groupby('page'):
 pm=np.zeros(runtime[page]['obs'].shape,np.uint8); idx=g.index[g.decision_pre_run_rule=='PRINT']; pm[full.loc[idx,'y'],full.loc[idx,'x']]=1; n,lab,stats,_=cv2.connectedComponentsWithStats(pm,8)
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<2:
   ys,xs=np.where(lab==cid); coords={(int(y),int(x)) for y,x in zip(ys,xs)}; exact=[i for i in idx if (int(full.at[i,'y']),int(full.at[i,'x'])) in coords]; full.loc[exact,'decision']='UNCERTAIN'; singleton_demotions+=len(exact)
full.to_parquet(TABLES/'all_eligible_oof_predictions.parquet',index=False); full.to_csv(TABLES/'all_eligible_oof_predictions.csv',index=False); print(full.decision.value_counts()); print('Singleton demotions:',singleton_demotions)


## 9. Whole-page assembly and direct integrity contract


In [ ]:
page_integrity=[]
for page,g in full.groupby('page'):
 d=WHOLE/page; d.mkdir(parents=True,exist_ok=False); z=runtime[page]; observed=z['obs']; baseline=z['baseline']; mask=z['mask']; out=baseline.copy(); sel=g[g.decision=='PRINT']
 if len(sel): out[sel.y.to_numpy(int),sel.x.to_numpy(int)]=observed[sel.y.to_numpy(int),sel.x.to_numpy(int)]
 print_mask=np.zeros(mask.shape,bool)
 if len(sel): print_mask[sel.y.to_numpy(int),sel.x.to_numpy(int)]=True
 outside=int(np.sum(out[~mask]!=observed[~mask])); nonprint=int(np.sum(out[~print_mask]!=baseline[~print_mask])); changed=(out!=baseline); wrong=int(np.sum(changed&(out!=observed)))
 if outside or nonprint or wrong: raise RuntimeError(f'{page}: page contract failed outside={outside}, nonprint={nonprint}, changed_not_observed={wrong}')
 cv2.imwrite(str(d/'observed.png'),observed); cv2.imwrite(str(d/'v7_8.png'),baseline); cv2.imwrite(str(d/'v8_2.png'),out); diff=np.zeros_like(observed); diff[changed]=255; cv2.imwrite(str(d/'difference.png'),diff); dm=np.zeros_like(observed); dm[g.y[g.decision=='ANNOTATION'],g.x[g.decision=='ANNOTATION']]=85; dm[g.y[g.decision=='UNCERTAIN'],g.x[g.decision=='UNCERTAIN']]=170; dm[g.y[g.decision=='PRINT'],g.x[g.decision=='PRINT']]=255; cv2.imwrite(str(d/'decision_map.png'),dm)
 page_integrity.append({'page':page,'source_group':g.source_group.iloc[0],'eligible':len(g),'print_pixels':len(sel),'outside_mask_drift':outside,'non_print_v78_regression':nonprint,'changed_pixels_not_equal_observed':wrong,'observed_array_sha256':array_hash(observed),'v78_array_sha256':array_hash(baseline),'v82_array_sha256':array_hash(out)})
pd.DataFrame(page_integrity).to_csv(AUDIT/'page_integrity.csv',index=False); print('Whole pages:',len(page_integrity))


## 10. Truth scoring and page/source/component evidence


In [ ]:
scored=full[full.truth.isin([0,1])].copy(); selected=scored.decision=='PRINT'; tp=int(np.sum(selected&(scored.truth==1))); fp=int(np.sum(selected&(scored.truth==0))); selected_scored=int(selected.sum()); truth_uncertain_prints=full[(full.decision=='PRINT')&(full.truth==2)].copy(); truth_uncertain_prints.to_csv(TABLES/'truth_uncertain_prints.csv',index=False)
component_rows=[]
for page,g in full.groupby('page'):
 z=runtime[page]; n,labels,stats,_=cv2.connectedComponentsWithStats(z['mask'].astype(np.uint8),8); sel=g[g.decision=='PRINT']
 if not len(sel): continue
 cids=sorted(set(labels[sel.y.to_numpy(int),sel.x.to_numpy(int)].tolist())-{0})
 for cid in cids:
  m=labels[sel.y.to_numpy(int),sel.x.to_numpy(int)]==cid; q=sel.loc[m]; ctp=int((q.truth==1).sum()); cfp=int((q.truth==0).sum()); cu=int((q.truth==2).sum()); catastrophic=bool(cfp>=50 or (cfp/max(ctp,1)>1 and cfp>=20)); component_rows.append({'page':page,'source_group':meta.loc[page,'source_group'],'component_id':int(cid),'mask_area':int(stats[cid,cv2.CC_STAT_AREA]),'selected':len(q),'true_print':ctp,'false_annotation':cfp,'truth_uncertain':cu,'false_true_ratio':cfp/max(ctp,1),'utility':ctp-5*cfp-(500 if catastrophic else 0),'catastrophic':catastrophic,'x':int(stats[cid,cv2.CC_STAT_LEFT]),'y':int(stats[cid,cv2.CC_STAT_TOP]),'w':int(stats[cid,cv2.CC_STAT_WIDTH]),'h':int(stats[cid,cv2.CC_STAT_HEIGHT])})
components=pd.DataFrame(component_rows) if component_rows else pd.DataFrame(columns=['page','source_group','component_id','mask_area','selected','true_print','false_annotation','truth_uncertain','false_true_ratio','utility','catastrophic','x','y','w','h']); components.to_csv(TABLES/'selected_components.csv',index=False); components[components.catastrophic==True].to_csv(TABLES/'catastrophic_components.csv',index=False); cat=int((components.catastrophic==True).sum())
page_rows=[]
for page,g in full.groupby('page'):
 q=g[g.truth.isin([0,1])]; s=q.decision=='PRINT'; pt=int(np.sum(s&(q.truth==1))); pf=int(np.sum(s&(q.truth==0))); pc=int(((components.page==page)&(components.catastrophic==True)).sum()); page_rows.append({'page':page,'source_group':g.source_group.iloc[0],'eligible':len(g),'selected':int((g.decision=='PRINT').sum()),'true_print':pt,'false_annotation':pf,'truth_uncertain_print':int(((g.decision=='PRINT')&(g.truth==2)).sum()),'false_true_ratio':pf/max(pt,1),'false_preservation_rate':pf/max(int(s.sum()),1),'utility':pt-5*pf-500*pc,'catastrophic_components':pc,'print_coverage':float((g.decision=='PRINT').mean()),'uncertain_rate':float((g.decision=='UNCERTAIN').mean())})
page_df=pd.DataFrame(page_rows); page_df.to_csv(TABLES/'results_by_page.csv',index=False); group_df=page_df.groupby('source_group',as_index=False).agg(pages=('page','nunique'),eligible=('eligible','sum'),selected=('selected','sum'),true_print=('true_print','sum'),false_annotation=('false_annotation','sum'),truth_uncertain_print=('truth_uncertain_print','sum'),utility=('utility','sum'),catastrophic_components=('catastrophic_components','sum')); group_df['false_true_ratio']=group_df.false_annotation/np.maximum(group_df.true_print,1); group_df['false_preservation_rate']=group_df.false_annotation/np.maximum(group_df.selected,1); group_df.to_csv(TABLES/'results_by_source_group.csv',index=False)
summary={'eligible_pixels':int(len(full)),'selected_print_pixels':int((full.decision=='PRINT').sum()),'true_print':tp,'false_annotation':fp,'truth_uncertain_print':int(len(truth_uncertain_prints)),'false_true_ratio':fp/max(tp,1),'false_preservation_rate':fp/max(selected_scored,1),'wilson95_upper_false_preservation':wilson_upper(fp,selected_scored),'utility':tp-5*fp-500*cat,'catastrophic_components':cat,'benefiting_pages':int(page_df.loc[page_df.true_print>=10,'page'].nunique()),'benefiting_source_groups':int(page_df.loc[page_df.true_print>=10,'source_group'].nunique()),'print_coverage':float((full.decision=='PRINT').mean()),'uncertain_rate':float((full.decision=='UNCERTAIN').mean())}; print(json.dumps(summary,indent=2))


## 11. Page-clustered bootstrap and development evidence flags


In [ ]:
rng=np.random.default_rng(SEED); unique_pages=page_df.page.to_numpy(); indexed=page_df.set_index('page'); boot=[]
for _ in range(BOOTSTRAP_RESAMPLES):
 q=indexed.loc[rng.choice(unique_pages,size=len(unique_pages),replace=True)]; bt=int(q.true_print.sum()); bf=int(q.false_annotation.sum()); bn=bt+bf; bc=int(q.catastrophic_components.sum()); boot.append((bt-5*bf-500*bc,bf/max(bt,1),bf/max(bn,1)))
b=np.asarray(boot,float); summary['bootstrap_utility_95_low']=float(np.quantile(b[:,0],.025)); summary['bootstrap_utility_95_high']=float(np.quantile(b[:,0],.975)); summary['bootstrap_false_true_95_high']=float(np.quantile(b[:,1],.975)); summary['bootstrap_false_preservation_95_high']=float(np.quantile(b[:,2],.975))
gates={'D1_integrity':bool(np.isfinite(raw_all).all() and np.isfinite(p_all).all()),'D2_immutability':bool(pd.DataFrame(page_integrity)[['outside_mask_drift','non_print_v78_regression','changed_pixels_not_equal_observed']].to_numpy().sum()==0),'D3_catastrophic_safety':bool(cat==0),'D4_false_true':bool(summary['false_true_ratio']<=.10 and summary['bootstrap_false_true_95_high']<=.20),'D5_pixel_safety':bool(summary['false_preservation_rate']<=.005 and summary['wilson95_upper_false_preservation']<=.01),'D6_positive_utility':bool(summary['utility']>0 and summary['bootstrap_utility_95_low']>0),'D7_nontrivial_benefit':bool(tp>=500 and int((page_df.true_print>=10).sum())>=10 and summary['benefiting_source_groups']>=3),'D8_source_stability':bool(((group_df.false_true_ratio<=.25)|(group_df.pages<3)).all() and (group_df.catastrophic_components==0).all()),'D9_sparse_hard_negative':'SEPARATE_CHALLENGE_EVIDENCE_NOT_IMPLEMENTED_IN_PRIMARY_V8_2_RUN','D10_uncertainty_reported':True,'D11_baseline_dominance':'NOT_RETESTED_PRIMARY_COMPARATOR_IS_V7_8','D12_visual_audit':'WHOLE_PAGE_OUTPUTS_GENERATED_REVIEW_NOT_COMPLETED'}; summary['gates']=gates; (OUT/'development_summary.json').write_text(json.dumps(summary,indent=2)); print(json.dumps(gates,indent=2))


## 12. Whole-page visual audit queues


In [ ]:
page_df.sort_values(['false_annotation','selected'],ascending=False).head(50).to_csv(TABLES/'visual_audit_false_heavy_pages.csv',index=False); page_df.sort_values(['true_print','utility'],ascending=False).head(50).to_csv(TABLES/'visual_audit_beneficial_pages.csv',index=False); page_df.sort_values('uncertain_rate',ascending=False).head(50).to_csv(TABLES/'visual_audit_uncertain_pages.csv',index=False)
def make_sheet(rows,name,max_pages=12):
 rows=rows.head(max_pages)
 if len(rows)==0:return
 fig,axes=plt.subplots(len(rows),2,figsize=(12,max(4,6*len(rows))),squeeze=False)
 for r,(_,row) in enumerate(rows.iterrows()):
  page=row.page; ims=[cv2.imread(str(WHOLE/page/'v7_8.png'),0),cv2.imread(str(WHOLE/page/'v8_2.png'),0)]
  for c,(im,title) in enumerate(zip(ims,[f'{page} V7.8',f'{page} V8.2'])): axes[r,c].imshow(im,cmap='gray',vmin=0,vmax=255); axes[r,c].set_title(title); axes[r,c].axis('off')
 plt.tight_layout(); plt.savefig(VISUAL/f'{name}.png',dpi=130,bbox_inches='tight'); plt.close(fig)
make_sheet(page_df.sort_values(['false_annotation','selected'],ascending=False),'false_heavy_pages'); make_sheet(page_df.sort_values(['true_print','utility'],ascending=False),'beneficial_pages'); make_sheet(page_df.sort_values('uncertain_rate',ascending=False),'uncertain_pages'); print('Visual audit:',VISUAL)


## 13. Self-contained experiment identity and summary


In [ ]:
environment={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'gpu':torch.cuda.get_device_name(0),'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()}; (OUT/'software_environment.json').write_text(json.dumps(environment,indent=2))
identity={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'8.2-development-whole-page','development_only':True,'promotion_authority':False,'v7_5_reference':str(V75),'manifest_snapshot':str(INPUTS/'development_score_manifest.csv'),'manifest_snapshot_sha256':sha256_file(INPUTS/'development_score_manifest.csv'),'fold_map':str(INPUTS/'fold_map.csv'),'fold_map_sha256':sha256_file(INPUTS/'fold_map.csv'),'channel_schema':str(INPUTS/'channel_schema.json'),'channel_schema_sha256':sha256_file(INPUTS/'channel_schema.json'),'pages':len(pages),'source_groups':int(manifest.source_group.nunique()),'stable_rows':len(stable),'eligible_rows':len(all_candidates),'seed':SEED,'patch':PATCH,'channels':CHANNELS,'batch':BATCH,'epochs':EPOCHS,'lr':LR,'weight_decay':WEIGHT_DECAY,'print_threshold':PRINT_THRESHOLD,'annotation_threshold':ANNOTATION_THRESHOLD,'print_margin':PRINT_MARGIN,'models':model_rows,'calibrators':calibrator_rows,'historical_v80_lock_required':False,'independent_data_used':False}; (OUT/'experiment_identity.json').write_text(json.dumps(identity,indent=2,default=str))
lines=['V8.2 DEVELOPMENT WHOLE-PAGE EXPERIMENT','',f'Run: {OUT}',f'Pages: {len(pages)}',f'Source groups: {manifest.source_group.nunique()}',f'Eligible pixels: {len(full)}',f'PRINT pixels: {int((full.decision=="PRINT").sum())}',f'True print: {tp}',f'False annotation: {fp}',f'Truth-uncertain PRINT: {len(truth_uncertain_prints)}',f'False/true ratio: {summary["false_true_ratio"]:.6f}',f'False-preservation rate: {summary["false_preservation_rate"]:.6f}',f'Utility: {summary["utility"]}',f'Catastrophic components: {cat}',f'PRINT coverage: {summary["print_coverage"]:.6f}',f'UNCERTAIN rate: {summary["uncertain_rate"]:.6f}','','DEVELOPMENT EVIDENCE FLAGS']+[f'- {k}: {v}' for k,v in gates.items()]+['','Fresh development experiment only. Freeze/replay separately before any untouched independent test.']; (OUT/'executive_summary.txt').write_text('\n'.join(lines)); print('\n'.join(lines)); print('\nV8.2 COMPLETE:',OUT)


## Outputs to review

Start with `executive_summary.txt`, `development_summary.json`, page/source-group result tables, `catastrophic_components.csv`, the `whole_pages/` V7.8 versus V8.2 images, and the visual-audit contact sheets.

If the run fails, do **not** suppress the error. Use `PAGE_ALLOWLIST` only for a deliberate small smoke test if you need to isolate a data/runtime issue.
